# SOXX 原始 NPORT-P 披露普通股集中度来源

13 份原始 XML 仅选 EC/Long/NS/USD/CORP，按原市值在披露普通股内部归一化；严格按 SEC filing 日后的下个 Qlib 会话映射 2019–2022 训练日，缺失前缀保留。无市场收益、拟合或账户。


In [1]:
import os,json,hashlib,xml.etree.ElementTree as ET,decimal,pickle,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_soxx_2019_2022_nport_equity_concentration_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
 return h.hexdigest()
assert plan['frozen'] and (card/'run_state.json').exists()
repair=json.loads((card/'CUSIP_repair_plan.json').read_text())
assert repair['frozen'] and sha(card/'CUSIP_diagnostic_result.json')==repair['diagnostic_sha256']
old_state=json.loads((card/'run_state.json').read_text())
final_repair=json.loads((card/'final_CUSIP_repair_plan.json').read_text())
assert final_repair['frozen'] and sha(card/'last_report_identity_diagnostic_result.json')==final_repair['diagnostic_sha256']
assert old_state['status']=='mechanical_repair_started' and old_state['XML_reads']==13 and not (card/'qlib_recorder').exists()
assert sha(root/plan['sources']['metadata_result_path'])==plan['sources']['metadata_result_sha256']
assert sha(root/plan['sources']['new_XML_source_result_path'])==plan['sources']['new_XML_source_result_sha256']
assert sha(root/plan['sources']['reused_2019_specimen_schema_result_path'])==plan['sources']['reused_2019_specimen_schema_sha256']
source_state=json.loads((root/plan['sources']['new_XML_source_result_path']).read_text());assert source_state['nas_remote_and_scoped_restore_verified']
state={'status':'final_source_pass_started','XML_reads_before_repair':16,'XML_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
Dnum=decimal.Decimal;local=lambda t:t.rsplit('}',1)[-1]
reports=[];member_rows=[]
for source in plan['sources']['original_XML']:
 p=root/source['raw_path'];assert sha(p)==source['raw_sha256'],('raw drift',source['accession'])
 state['XML_reads']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 xml=ET.fromstring(p.read_bytes())
 def text(tag):return sorted({(e.text or '').strip() for e in xml.iter() if local(e.tag)==tag and (e.text or '').strip()})
 assert text('cik')==['0001100663'] and text('seriesId')==['S000004354'] and text('classId')==['C000012084']
 assert text('repPdDate')==[source['report_period']]
 net_assets=Dnum(text('netAssets')[0]);assert net_assets>0
 holdings=[e for e in xml.iter() if local(e.tag)=='invstOrSec']
 def field(e,name):return next(((q.text or '').strip() for q in e if local(q.tag)==name),'')
 categories={x:sum(field(e,'assetCat')==x for e in holdings) for x in ['EC','DE','STIV']}
 assert sum(categories.values())==len(holdings) and categories['EC']==30
 equity=[];cusips=set();names=set();pct_max=Dnum(0)
 for e in holdings:
  if field(e,'assetCat')!='EC':continue
  assert field(e,'payoffProfile')=='Long' and field(e,'units')=='NS' and field(e,'curCd')=='USD' and field(e,'issuerCat')=='CORP'
  cusip=field(e,'cusip');name=field(e,'name');assert name and name.casefold() not in names
  names.add(name.casefold())
  if cusip in {'000000000','N/A'}:identifier_status='placeholder_CUSIP_name_only'
  else:
   assert cusip and cusip not in cusips
   cusips.add(cusip);identifier_status='original_unique_CUSIP'
  balance=Dnum(field(e,'balance'));value=Dnum(field(e,'valUSD'));pct=Dnum(field(e,'pctVal'))
  assert balance>0 and value>0 and pct>0
  delta=abs(pct-Dnum(100)*value/net_assets);pct_max=max(pct_max,delta)
  assert delta<=Dnum('0.000001')
  equity.append({'accession':source['accession'],'report_period':source['report_period'],'cusip_original':cusip,'identifier_status':identifier_status,'security_original':name,'shares_original':str(balance),'fair_value_usd_original':str(value),'pct_net_assets_original':str(pct),'source_xml_sha256':source['raw_sha256']})
 total=sum((Dnum(x['fair_value_usd_original']) for x in equity),Dnum(0));assert total>0 and len(equity)==30
 weights=[Dnum(x['fair_value_usd_original'])/total for x in equity]
 assert abs(sum(weights)-Dnum(1))<Dnum('1e-25')
 reports.append({'accession':source['accession'],'report_period':source['report_period'],'filed':source['filed'],'net_assets_usd_original':str(net_assets),'EC_value_usd_original':str(total),'EC_rows':30,'other_asset_category_counts':{'DE':categories['DE'],'STIV':categories['STIV']},'max_pct_net_assets_closure_error_pp':str(pct_max),'top5_disclosed_EC_value_share':float(sum(sorted(weights,reverse=True)[:5])),'HHI_disclosed_EC_value':float(sum(w*w for w in weights)),'source_xml_sha256':source['raw_sha256']})
 member_rows.extend(equity)
assert len(reports)==13 and len(member_rows)==390
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(root/plan['calendar_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
calendar=pd.DatetimeIndex(D.calendar(start_time='2019-01-02',end_time='2023-01-10',freq='day'))
events=pd.DataFrame(reports).sort_values('filed').reset_index(drop=True)
events['filed_date']=pd.to_datetime(events.filed)
events['effective_signal_date']=[calendar[int(calendar.searchsorted(d,side='right'))] for d in events.filed_date]
assert events.effective_signal_date.is_unique and (events.effective_signal_date>events.filed_date).all()
signals=calendar[(calendar>=pd.Timestamp('2019-01-02'))&(calendar<=pd.Timestamp('2022-12-30'))]
mapped=pd.merge_asof(pd.DataFrame({'signal_date':signals}),events[['effective_signal_date','report_period','filed_date','top5_disclosed_EC_value_share','HHI_disclosed_EC_value']],left_on='signal_date',right_on='effective_signal_date',direction='backward')
mapped['known']=mapped.report_period.notna();mapped['source_age_days']=(mapped.signal_date-mapped.filed_date).dt.days
assert mapped.signal_date.is_unique and len(mapped)==len(signals) and mapped.loc[mapped.known,'source_age_days'].ge(1).all()
assert not mapped.loc[mapped.signal_date<mapped.effective_signal_date.min(),'known'].any()
frame=mapped.set_index('signal_date')[['top5_disclosed_EC_value_share','HHI_disclosed_EC_value','known','source_age_days']]
frame.index=pd.MultiIndex.from_arrays([signals,['SOXL_SOXX_CASH']*len(signals)],names=['datetime','instrument'])
static=pd.concat({'feature':frame[['top5_disclosed_EC_value_share','HHI_disclosed_EC_value']],'meta':frame[['known','source_age_days']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'late_train':('2019-01-02','2022-12-30')})
prepared=dataset.prepare('late_train',col_set=['feature','meta']);assert len(prepared)==len(signals)
assert int(mapped.known.sum())==int(prepared[('meta','known')].sum())
audit={'status':'NPORT_original_13_report_EC_concentration_source_events_Qlib_mapped_no_fit','source_plan_sha256':sha(card/'plan.json'),'report_events':len(reports),'member_rows':len(member_rows),'placeholder_CUSIP_rows':sum(x['identifier_status']=='placeholder_CUSIP_name_only' for x in member_rows),'XML_reads_total':16+state['XML_reads'],'late_train_Qlib_sessions':len(signals),'known_signal_sessions':int(mapped.known.sum()),'unknown_prefix_sessions':int((~mapped.known).sum()),'first_effective_signal':str(events.effective_signal_date.min().date()),'last_effective_signal':str(events.effective_signal_date.max().date()),'max_source_age_days':int(mapped.source_age_days.max()),'top5_source_bounds':[min(x['top5_disclosed_EC_value_share'] for x in reports),max(x['top5_disclosed_EC_value_share'] for x in reports)],'HHI_source_bounds':[min(x['HHI_disclosed_EC_value'] for x in reports),max(x['HHI_disclosed_EC_value'] for x in reports)],'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'full_Stage1_training_coverage_qualified':False}
(card/'source_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
dataset.config(dump_all=True,recursive=True)
assert len(pickle.loads(pickle.dumps(dataset)).prepare('late_train',col_set=['feature','meta']))==len(signals)
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['phase'],uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_hashes={x['accession']:x['raw_sha256'] for x in plan['sources']['original_XML']})
 rec.save_objects(artifact_path='analysis',source_events=events,member_rows=member_rows,mapped_signals=mapped,source_audit=audit)
 rec.save_objects(artifact_path='metadata',dataset=dataset)
 rec.log_metrics(source_report_events=len(reports),late_train_sessions=len(signals),known_source_sessions=int(mapped.known.sum()),model_fits=0,native_accounts=0)
 recorder_id=rec.id;experiment_id=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=recorder_id,experiment_id=experiment_id)
 assert len(checked.load_object('metadata/dataset').prepare('late_train',col_set=['feature','meta']))==len(signals)
 assert len(checked.load_object('analysis/source_events'))==13
settings=_settings(root);binding=card/'nas_binding.json'
binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['phase'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
with tempfile.TemporaryDirectory(prefix='soxx-concentration-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:
  run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:
  names=[m.name for m in source.getmembers() if m.isfile()]
 assert any(name.endswith('/artifacts/metadata/dataset') for name in names)
summary={'experiment':'stage1_soxx_2019_2022_nport_equity_concentration_20261023','decision':'late_training_original_NPORT_reported_EC_concentration_source_qualified_early_history_pending','objective_achieved':False,'plan':'plan.json','source_audit':'source_audit.json','mechanical_repairs':['CUSIP_repair_plan.json','final_CUSIP_repair_plan.json'],'failed_attempts':['attempt1_failure.json','attempt2_failure.json'],'report_events':13,'member_rows':390,'model_fits':0,'native_accounts':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'artifact_policy':plan['artifact_policy']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['status']='completed';state['XML_reads_total']=16+state['XML_reads'];state['recorder_id']=recorder_id;state['nas_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({k:audit[k] for k in ('report_events','late_train_Qlib_sessions','known_signal_sessions','unknown_prefix_sessions','first_effective_signal','max_source_age_days')})
print({'recorder_id':recorder_id,'nas_sha256':evidence['sha256']})


{'report_events': 13, 'late_train_Qlib_sessions': 1008, 'known_signal_sessions': 780, 'unknown_prefix_sessions': 228, 'first_effective_signal': '2019-11-26', 'max_source_age_days': 95}
{'recorder_id': '7f2b90c01f3c4820a6173a113018adf3', 'nas_sha256': '720ed1ed5437451dd39a2b3ce324c4f503062177339bde861561a8050feb2393'}


## 中文结论

13 份原始 SOXX NPORT-P 报告各有 30 条 EC/Long/NS/USD/CORP 普通股，共 390 条；逐行 `pctVal` 与原始 `valUSD/netAssets` 闭合。普通股内部原市值权重的前五占比 0.350918–0.412843、HHI 0.047130–0.054053。两种原文 CUSIP 占位符 `000000000`/`N/A` 的 33 行不删除、不倒填身份；两次机械失败及修复保留。2019–2022 Qlib 日历 1,008 日全留，首份公开前 228 日未知，后 780 日已知，最大申报日龄 95 天。DatasetH/Recorder `7f2b90c01f3c4820a6173a113018adf3` 及 NAS SHA-256 `720ed1ed5437451dd39a2b3ce324c4f503062177339bde861561a8050feb2393` 读回通过。0 市场收益、0 拟合、0 账户；2011–2018 与 2019 前缀来源缺口未补，不能声称 SOXX 超额。
